# Energy Forecasting Model Training (LSTM)
**Datasets:**
- Solar: Time Series Forecasting of Solar Energy — https://www.kaggle.com/datasets/chaitanyakumar12/time-series-forecasting-of-solar-energy
- Wind: Wind Power Generation Data — https://www.kaggle.com/datasets/mubashirrahim/wind-power-generation-data-forecasting

**Output:** `forecast_model.pt`  
**Model:** PyTorch LSTM (multi-output: solar kWh + wind kWh)  
**Input:** sequence of 30 days [solar_irradiance, wind_speed] → **Output:** next 30 days [solar_kwh, wind_kwh]

In [ ]:
import numpy as np
import pandas as pd
import glob
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

SEQ_LEN = 30   # input: 30 days of history
PRED_LEN = 30  # output: 30 days forecast
BATCH_SIZE = 64
EPOCHS = 50
LR = 1e-3

## 1. Load & Prepare Solar Data
Add both datasets to this notebook on Kaggle.

In [ ]:
# --- Solar dataset ---
solar_files = glob.glob('/kaggle/input/**/*solar*/*.csv', recursive=True) or \
              glob.glob('/kaggle/input/*solar*/**/*.csv', recursive=True)
print('Solar files:', solar_files)

solar_df = pd.read_csv(solar_files[0])
print(solar_df.shape)
print(solar_df.columns.tolist())
solar_df.head()

In [ ]:
# Detect datetime and power/irradiance columns
def find_col(df, keywords):
    for kw in keywords:
        match = [c for c in df.columns if kw.lower() in c.lower()]
        if match: return match[0]
    return None

solar_time_col  = find_col(solar_df, ['date', 'time', 'datetime', 'timestamp'])
solar_power_col = find_col(solar_df, ['power', 'generation', 'energy', 'kwh', 'output', 'ac_power'])
solar_irr_col   = find_col(solar_df, ['irradiation', 'irradiance', 'ghi', 'radiation'])

print(f'Time: {solar_time_col}, Power: {solar_power_col}, Irradiance: {solar_irr_col}')

In [ ]:
solar_df[solar_time_col] = pd.to_datetime(solar_df[solar_time_col])
solar_df['date'] = solar_df[solar_time_col].dt.date

agg = {}
if solar_power_col: agg['solar_kwh'] = (solar_power_col, 'sum')
if solar_irr_col:   agg['solar_irradiance'] = (solar_irr_col, 'mean')

solar_daily = solar_df.groupby('date').agg(**agg).reset_index()
solar_daily = solar_daily.sort_values('date').reset_index(drop=True)

# If no power column, use irradiance as proxy for solar_kwh
if 'solar_kwh' not in solar_daily.columns:
    solar_daily['solar_kwh'] = solar_daily['solar_irradiance'] * 0.05
if 'solar_irradiance' not in solar_daily.columns:
    solar_daily['solar_irradiance'] = solar_daily['solar_kwh'] * 20

print(solar_daily.shape)
solar_daily.head()

## 2. Load & Prepare Wind Data

In [ ]:
wind_files = glob.glob('/kaggle/input/**/*wind*/*.csv', recursive=True) or \
             glob.glob('/kaggle/input/*wind*/**/*.csv', recursive=True)
print('Wind files:', wind_files)

wind_df = pd.read_csv(wind_files[0])
print(wind_df.shape)
print(wind_df.columns.tolist())
wind_df.head()

In [ ]:
wind_time_col  = find_col(wind_df, ['date', 'time', 'datetime', 'timestamp'])
wind_power_col = find_col(wind_df, ['power', 'active_power', 'lv_activepower', 'output', 'generation'])
wind_speed_col = find_col(wind_df, ['wind_speed', 'ws', 'speed'])

print(f'Time: {wind_time_col}, Power: {wind_power_col}, Speed: {wind_speed_col}')

wind_df[wind_time_col] = pd.to_datetime(wind_df[wind_time_col])
wind_df['date'] = wind_df[wind_time_col].dt.date

wind_agg = {}
if wind_power_col: wind_agg['wind_kwh'] = (wind_power_col, 'sum')
if wind_speed_col: wind_agg['wind_speed'] = (wind_speed_col, 'mean')

wind_daily = wind_df.groupby('date').agg(**wind_agg).reset_index()
wind_daily = wind_daily.sort_values('date').reset_index(drop=True)

if 'wind_kwh' not in wind_daily.columns:
    wind_daily['wind_kwh'] = wind_daily['wind_speed'] ** 3 * 0.01
if 'wind_speed' not in wind_daily.columns:
    wind_daily['wind_speed'] = (wind_daily['wind_kwh'] / 0.01) ** (1/3)

print(wind_daily.shape)
wind_daily.head()

## 3. Build Sequences

In [ ]:
# Align both series to same length (use shorter one)
min_len = min(len(solar_daily), len(wind_daily))
solar_daily = solar_daily.iloc[:min_len]
wind_daily  = wind_daily.iloc[:min_len]

# Input features: [solar_irradiance, wind_speed] — what our platform collects per site
# Output targets: [solar_kwh, wind_kwh] — what we want to forecast
X_raw = np.column_stack([solar_daily['solar_irradiance'].values, wind_daily['wind_speed'].values])
y_raw = np.column_stack([solar_daily['solar_kwh'].values, wind_daily['wind_kwh'].values])

# Normalize
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()
X_scaled = scaler_X.fit_transform(X_raw)
y_scaled = scaler_y.fit_transform(y_raw)

print(f'X shape: {X_scaled.shape}, y shape: {y_scaled.shape}')

In [ ]:
def make_sequences(X, y, seq_len, pred_len):
    xs, ys = [], []
    for i in range(len(X) - seq_len - pred_len + 1):
        xs.append(X[i:i+seq_len])
        ys.append(y[i+seq_len:i+seq_len+pred_len])
    return np.array(xs), np.array(ys)

X_seq, y_seq = make_sequences(X_scaled, y_scaled, SEQ_LEN, PRED_LEN)
print(f'Sequences — X: {X_seq.shape}, y: {y_seq.shape}')

split = int(len(X_seq) * 0.8)
X_train, X_test = X_seq[:split], X_seq[split:]
y_train, y_test = y_seq[:split], y_seq[split:]

class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_loader = DataLoader(SeqDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(SeqDataset(X_test, y_test),  batch_size=BATCH_SIZE)

## 4. Define & Train LSTM

In [ ]:
class ForecastLSTM(nn.Module):
    def __init__(self, input_size=2, hidden_size=64, num_layers=2, pred_len=30, output_size=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_size, pred_len * output_size)
        self.pred_len = pred_len
        self.output_size = output_size

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])  # last timestep
        return out.view(-1, self.pred_len, self.output_size)

model = ForecastLSTM().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.MSELoss()
print(model)

In [ ]:
for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    if epoch % 10 == 0:
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                val_loss += criterion(model(xb), yb).item()
        print(f'Epoch {epoch:3d} | Train Loss: {train_loss/len(train_loader):.6f} | Val Loss: {val_loss/len(test_loader):.6f}')

## 5. Evaluate

In [ ]:
model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        preds = model(xb.to(DEVICE)).cpu().numpy()
        all_preds.append(preds)
        all_true.append(yb.numpy())

all_preds = np.concatenate(all_preds).reshape(-1, 2)
all_true  = np.concatenate(all_true).reshape(-1, 2)

# Inverse transform
preds_inv = scaler_y.inverse_transform(all_preds)
true_inv  = scaler_y.inverse_transform(all_true)

print(f'Solar MAE: {mean_absolute_error(true_inv[:,0], preds_inv[:,0]):.2f} kWh')
print(f'Wind  MAE: {mean_absolute_error(true_inv[:,1], preds_inv[:,1]):.2f} kWh')

## 6. Save Model

In [ ]:
# Move to CPU before saving so it loads on any machine
model.cpu()
torch.save(model, 'forecast_model.pt')
print('Saved: forecast_model.pt')

# Verify
loaded = torch.load('forecast_model.pt', map_location='cpu')
loaded.eval()
dummy = torch.zeros(1, SEQ_LEN, 2)  # batch=1, seq=30, features=2
out = loaded(dummy)
print(f'Output shape: {out.shape}')  # expected: [1, 30, 2]